# Tutorial: Membangun WebGIS dengan VS Code

Notebook ini menjelaskan langkah bertahap untuk membangun WebGIS sederhana menggunakan **HTML**, **CSS**, **JavaScript**, **MapLibre GL JS**, dan file **GeoJSON**.

Output akhir tutorial ini adalah aplikasi WebGIS yang memiliki:

1. halaman `index.html`;
2. tampilan peta dengan MapLibre;
3. satu basemap **Light**;
4. data polygon dari `jumlah_penduduk_surabaya.geojson`;
5. header, judul, icon, dan logo sederhana;
6. panel filter berdasarkan kecamatan;
7. popup informasi ketika polygon diklik;
8. struktur project siap deploy ke GitHub Pages.


## 1. Membuka VS Code dan Membuat Project Baru

Langkah pertama adalah membuat satu folder khusus untuk project WebGIS. Folder ini akan menjadi tempat semua file aplikasi disimpan, sehingga struktur kerja lebih rapi dan mudah dipindahkan ke GitHub.

Tahapan di VS Code:

1. Buka **Visual Studio Code**.
2. Pilih menu **File > Open Folder**.
3. Buat folder baru, misalnya `webgis-surabaya`.
4. Klik **Open**.
5. Di dalam folder tersebut, buat struktur file berikut:

```text
webgis-surabaya/
├── index.html
├── css/
│   └── style.css
├── js/
│   └── app.js
└── data/
    └── jumlah_penduduk_surabaya.geojson
```

File `index.html` adalah kerangka halaman web, `style.css` mengatur tampilan visual, `app.js` mengatur logika peta, dan folder `data` menyimpan file GeoJSON.

## 2. Menyiapkan Struktur Project secara Otomatis dari Notebook

Cell berikut dapat dijalankan untuk membuat struktur project secara otomatis. Apabila Anda bekerja langsung di VS Code, kode ini tidak wajib dijalankan; cukup buat folder dan file secara manual sesuai struktur di atas.

In [ ]:
from pathlib import Path
import shutil

project = Path("webgis-surabaya")
(project / "css").mkdir(parents=True, exist_ok=True)
(project / "js").mkdir(parents=True, exist_ok=True)
(project / "data").mkdir(parents=True, exist_ok=True)

# Jika notebook dijalankan di folder yang sama dengan file GeoJSON,
# salin file GeoJSON ke folder data.
source_geojson = Path("jumlah_penduduk_surabaya.geojson")
target_geojson = project / "data" / "jumlah_penduduk_surabaya.geojson"

if source_geojson.exists():
    shutil.copy(source_geojson, target_geojson)

print("Struktur folder project selesai dibuat.")

## 3. Membuat File `index.html` sebagai Kerangka WebGIS

File `index.html` berfungsi sebagai kerangka utama aplikasi. Di dalam file ini kita menyiapkan:

- metadata halaman web;
- pemanggilan library **MapLibre GL JS**;
- pemanggilan file CSS;
- header aplikasi;
- sidebar untuk kontrol peta;
- elemen `<div id="map"></div>` sebagai tempat peta ditampilkan;
- pemanggilan file JavaScript `app.js`.

Bagian paling penting adalah `<div id="map"></div>`, karena MapLibre akan mencari elemen ini sebagai container peta.

In [ ]:
%%writefile webgis-surabaya/index.html
<!DOCTYPE html>
<html lang="id">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />

  <title>WebGIS Jumlah Penduduk Surabaya</title>

  <!-- MapLibre GL JS: library utama untuk menampilkan peta web -->
  <link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet" />
  <script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>

  <!-- CSS buatan sendiri -->
  <link rel="stylesheet" href="./css/style.css" />
</head>

<body>
  <!-- Header aplikasi -->
  <header class="app-header">
    <div class="brand">
      <div class="logo">W</div>
      <div>
        <h1>WebGIS</h1>
        <p>Contoh WebGIS</p>
      </div>
    </div>
  </header>

  <!-- Layout utama: sidebar + peta -->
  <main class="app-layout">
    <aside class="sidebar">
      <section class="panel">
        <h2>Kontrol Peta</h2>
        <p class="helper-text">
          Gunakan panel ini untuk memfilter wilayah berdasarkan kecamatan.
        </p>

        <label for="kecamatanSelect">Filter Kecamatan</label>
        <select id="kecamatanSelect">
          <option value="all">Semua Kecamatan</option>
        </select>

        <button id="resetFilterBtn">Reset Filter</button>
      </section>

      <section class="panel">
        <h2>Legenda</h2>
        <div class="legend-item"><span style="background:#fff7bc"></span> Rendah</div>
        <div class="legend-item"><span style="background:#fec44f"></span> Sedang</div>
        <div class="legend-item"><span style="background:#d95f0e"></span> Tinggi</div>
        <div class="legend-item"><span style="background:#7f2704"></span> Sangat Tinggi</div>
      </section>
    </aside>

    <section class="map-container">
      <div id="map"></div>
    </section>
  </main>

  <!-- JavaScript aplikasi -->
  <script src="./js/app.js"></script>
</body>
</html>



### Penjelasan Singkat `index.html`

Kode `index.html` memiliki tiga blok utama. Blok `<head>` memuat konfigurasi halaman, link CSS MapLibre, script MapLibre, dan file `style.css`. Blok `<header>` membentuk identitas aplikasi berupa logo, judul, dan label sederhana. Blok `<main>` membagi layar menjadi dua area, yaitu sidebar kontrol dan area peta.

Elemen sidebar hanya berisi filter kecamatan, tombol reset, dan legenda. Basemap tidak lagi dibuat sebagai dropdown karena aplikasi memakai satu basemap Light saja. Setiap elemen penting diberi `id`, seperti `kecamatanSelect`, `resetFilterBtn`, dan `map`, agar dapat dipanggil oleh JavaScript. File `app.js` dipanggil paling bawah agar elemen HTML sudah terbaca terlebih dahulu sebelum logika peta dijalankan.


## 4. Membuat File `style.css` untuk Visualisasi Tampilan

File `style.css` mengatur tampilan aplikasi agar WebGIS tidak hanya berfungsi, tetapi juga nyaman digunakan. CSS ini mengatur header, sidebar, panel kontrol, legenda, area peta, popup, dan tampilan responsif untuk layar kecil.

In [ ]:
%%writefile webgis-surabaya/css/style.css
/* Reset sederhana agar layout konsisten di browser */
* {
  box-sizing: border-box;
}

body {
  margin: 0;
  font-family: Arial, Helvetica, sans-serif;
  background: #f8fafc;
  color: #0f172a;
}

/* Header bagian atas aplikasi */
.app-header {
  height: 72px;
  padding: 0 24px;
  background: #ffffff;
  border-bottom: 1px solid #e2e8f0;
  display: flex;
  align-items: center;
  justify-content: space-between;
}

.brand {
  display: flex;
  align-items: center;
  gap: 14px;
}

.logo {
  width: 42px;
  height: 42px;
  border-radius: 14px;
  background: #2563eb;
  color: white;
  display: flex;
  align-items: center;
  justify-content: center;
  font-weight: 800;
  box-shadow: 0 8px 20px rgba(37, 99, 235, 0.25);
}

.brand h1 {
  margin: 0;
  font-size: 18px;
  font-weight: 800;
}

.brand p {
  margin: 4px 0 0;
  font-size: 12px;
  color: #64748b;
}

.header-actions {
  display: flex;
  gap: 8px;
}

.badge {
  padding: 8px 10px;
  border-radius: 999px;
  background: #eff6ff;
  color: #1d4ed8;
  font-size: 11px;
  font-weight: 700;
}

/* Layout utama */
.app-layout {
  height: calc(100vh - 72px);
  display: flex;
}

/* Sidebar kiri */
.sidebar {
  width: 330px;
  background: #ffffff;
  border-right: 1px solid #e2e8f0;
  padding: 18px;
  overflow-y: auto;
}

.panel {
  background: #f8fafc;
  border: 1px solid #e2e8f0;
  border-radius: 18px;
  padding: 16px;
  margin-bottom: 16px;
}

.panel h2 {
  margin: 0 0 10px;
  font-size: 14px;
  font-weight: 800;
}

.helper-text {
  font-size: 12px;
  line-height: 1.6;
  color: #475569;
}

label {
  display: block;
  margin: 14px 0 6px;
  font-size: 11px;
  font-weight: 800;
  text-transform: uppercase;
  color: #475569;
}

select,
button {
  width: 100%;
  min-height: 40px;
  border-radius: 12px;
  border: 1px solid #cbd5e1;
  background: #ffffff;
  padding: 0 12px;
  font-size: 13px;
}

button {
  margin-top: 12px;
  border: none;
  background: #2563eb;
  color: #ffffff;
  font-weight: 800;
  cursor: pointer;
}

button:hover {
  background: #1d4ed8;
}

.legend-item {
  display: flex;
  align-items: center;
  gap: 10px;
  font-size: 12px;
  margin: 8px 0;
}

.legend-item span {
  width: 24px;
  height: 14px;
  border-radius: 4px;
  border: 1px solid rgba(15, 23, 42, 0.15);
}

/* Area peta */
.map-container {
  flex: 1;
  position: relative;
}

#map {
  width: 100%;
  height: 100%;
}

/* Popup MapLibre */
.maplibregl-popup-content {
  border-radius: 14px;
  padding: 14px;
  box-shadow: 0 12px 30px rgba(15, 23, 42, 0.2);
}

.popup-title {
  font-weight: 800;
  margin-bottom: 6px;
}

.popup-row {
  font-size: 12px;
  margin: 4px 0;
  color: #334155;
}

/* Responsif untuk layar kecil */
@media (max-width: 800px) {
  .app-layout {
    flex-direction: column;
  }

  .sidebar {
    width: 100%;
    height: 310px;
    border-right: none;
    border-bottom: 1px solid #e2e8f0;
  }

  .map-container {
    height: calc(100vh - 72px - 310px);
  }

  .header-actions {
    display: none;
  }
}


### Penjelasan Singkat `style.css`

Bagian `.app-header` mengatur header agar berada di bagian atas dengan tinggi tetap 72 piksel. Bagian `.app-layout` memakai `display: flex`, sehingga sidebar dan peta dapat berdampingan. Bagian `.sidebar` mengatur panel kiri, sedangkan `#map` diberi `width: 100%` dan `height: 100%` agar peta memenuhi ruang yang tersedia.

Bagian `.maplibregl-popup-content` mengatur tampilan popup bawaan MapLibre agar lebih halus dan konsisten dengan desain dashboard. Media query `@media (max-width: 800px)` membuat tampilan lebih responsif ketika aplikasi dibuka di layar kecil. Ini penting karena WebGIS sering diakses dari laptop, tablet, atau layar browser dengan ukuran berbeda.

## 5. Membuat File `app.js` untuk Menampilkan Peta MapLibre

File `app.js` adalah pusat logika WebGIS. Di file ini kita melakukan beberapa hal penting:

1. mendefinisikan satu basemap Light;
2. menginisialisasi objek peta MapLibre;
3. membaca file GeoJSON lokal;
4. menambahkan source dan layer polygon;
5. membuat filter kecamatan;
6. membuat popup ketika polygon diklik;
7. mengatur tombol reset untuk mengembalikan tampilan awal.


In [ ]:
%%writefile webgis-surabaya/js/app.js
// ===============================
// 1. Konfigurasi basemap
// ===============================
// Basemap adalah peta dasar. Pada tutorial ini kita hanya memakai satu basemap Light.
// Dengan satu basemap, tampilan panel lebih sederhana dan tidak membutuhkan basemap switcher.
const BASEMAP = "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json";

// ===============================
// 2. Inisialisasi peta MapLibre
// ===============================
// container: id elemen HTML tempat peta ditampilkan.
// style: URL style basemap Light.
// center: koordinat pusat peta [longitude, latitude] Surabaya.
// zoom: tingkat kedekatan peta.
const map = new maplibregl.Map({
  container: "map",
  style: BASEMAP,
  center: [112.7521, -7.2575],
  zoom: 11
});

// Menambahkan tombol zoom dan kompas.
map.addControl(new maplibregl.NavigationControl(), "top-right");

// Variabel global untuk menyimpan data GeoJSON setelah dibaca.
let pendudukData = null;

// Nama source dan layer dibuat sebagai konstanta agar tidak salah ketik.
const SOURCE_ID = "penduduk-surabaya-source";
const FILL_LAYER_ID = "penduduk-surabaya-fill";
const LINE_LAYER_ID = "penduduk-surabaya-outline";

// ===============================
// 3. Membaca file GeoJSON lokal
// ===============================
// File berada di folder data, sehingga path-nya ./data/jumlah_penduduk_surabaya.geojson
async function loadGeoJSON() {
  try {
    const response = await fetch("./data/jumlah_penduduk_surabaya.geojson");

    if (!response.ok) {
      throw new Error(`HTTP error: ${response.status}`);
    }

    pendudukData = await response.json();

    addPendudukLayer();
    buildKecamatanFilter();
  } catch (error) {
    console.error("Gagal membaca GeoJSON:", error);
    alert("GeoJSON gagal dimuat. Pastikan file berada di folder data.");
  }
}

// ===============================
// 4. Menambahkan layer GeoJSON ke peta
// ===============================
function addPendudukLayer() {
  // Jika source atau layer sudah ada, hapus dulu agar tidak terjadi duplikasi.
  if (map.getLayer(FILL_LAYER_ID)) map.removeLayer(FILL_LAYER_ID);
  if (map.getLayer(LINE_LAYER_ID)) map.removeLayer(LINE_LAYER_ID);
  if (map.getSource(SOURCE_ID)) map.removeSource(SOURCE_ID);

  // Source adalah sumber data spasial yang akan dibaca MapLibre.
  map.addSource(SOURCE_ID, {
    type: "geojson",
    data: pendudukData
  });

  // Layer fill untuk menampilkan polygon kelurahan/kecamatan.
  map.addLayer({
    id: FILL_LAYER_ID,
    type: "fill",
    source: SOURCE_ID,
    paint: {
      // Warna polygon dibuat berdasarkan atribut jumlah_pdd.
      // Semakin besar jumlah penduduk, semakin gelap warnanya.
      "fill-color": [
        "interpolate",
        ["linear"],
        ["get", "jumlah_pdd"],
        0, "#fff7bc",
        10000, "#fec44f",
        25000, "#d95f0e",
        50000, "#7f2704"
      ],
      "fill-opacity": 0.72
    }
  });

  // Layer garis batas agar polygon lebih mudah dibaca.
  map.addLayer({
    id: LINE_LAYER_ID,
    type: "line",
    source: SOURCE_ID,
    paint: {
      "line-color": "#0f172a",
      "line-width": 0.7,
      "line-opacity": 0.65
    }
  });
}

// ===============================
// 5. Membuat dropdown filter kecamatan
// ===============================
function buildKecamatanFilter() {
  const select = document.getElementById("kecamatanSelect");

  // Kosongkan option lama agar tidak dobel jika fungsi terpanggil ulang.
  select.innerHTML = '<option value="all">Semua Kecamatan</option>';

  // Ambil semua nilai WADMKC dari fitur GeoJSON.
  const kecamatanList = pendudukData.features
    .map((feature) => feature.properties.WADMKC)
    .filter(Boolean);

  // Buat daftar unik dan urutkan alfabetis.
  const uniqueKecamatan = [...new Set(kecamatanList)].sort();

  // Tambahkan setiap kecamatan sebagai option di dropdown.
  uniqueKecamatan.forEach((namaKecamatan) => {
    const option = document.createElement("option");
    option.value = namaKecamatan;
    option.textContent = namaKecamatan;
    select.appendChild(option);
  });
}

// ===============================
// 6. Menerapkan filter layer
// ===============================
function applyKecamatanFilter(kecamatan) {
  if (kecamatan === "all") {
    map.setFilter(FILL_LAYER_ID, null);
    map.setFilter(LINE_LAYER_ID, null);
    return;
  }

  // Filter MapLibre: tampilkan fitur yang atribut WADMKC-nya sama dengan pilihan user.
  const filter = ["==", ["get", "WADMKC"], kecamatan];
  map.setFilter(FILL_LAYER_ID, filter);
  map.setFilter(LINE_LAYER_ID, filter);

  // Zoom otomatis ke wilayah kecamatan yang dipilih.
  zoomToKecamatan(kecamatan);
}

// ===============================
// 7. Zoom otomatis ke kecamatan terpilih
// ===============================
function zoomToKecamatan(kecamatan) {
  const selectedFeatures = pendudukData.features.filter(
    (feature) => feature.properties.WADMKC === kecamatan
  );

  if (selectedFeatures.length === 0) return;

  const bounds = new maplibregl.LngLatBounds();

  selectedFeatures.forEach((feature) => {
    extendBoundsByCoordinates(bounds, feature.geometry.coordinates);
  });

  map.fitBounds(bounds, {
    padding: 60,
    duration: 900
  });
}

// Fungsi rekursif untuk membaca koordinat Polygon atau MultiPolygon.
function extendBoundsByCoordinates(bounds, coordinates) {
  coordinates.forEach((coord) => {
    if (typeof coord[0] === "number" && typeof coord[1] === "number") {
      bounds.extend(coord);
    } else {
      extendBoundsByCoordinates(bounds, coord);
    }
  });
}

// ===============================
// 8. Popup ketika polygon diklik
// ===============================
function setupPopup() {
  // Popup akan muncul saat user klik polygon pada layer fill.
  map.on("click", FILL_LAYER_ID, (event) => {
    const props = event.features[0].properties;

    const popupHTML = `
      <div>
        <div class="popup-title">${props.NAMOBJ || "Tanpa Nama"}</div>
        <div class="popup-row"><b>Kecamatan:</b> ${props.WADMKC || "-"}</div>
        <div class="popup-row"><b>Jumlah Penduduk:</b> ${Number(props.jumlah_pdd || 0).toLocaleString("id-ID")} jiwa</div>
        <div class="popup-row"><b>Luas:</b> ${props.luas ? Number(props.luas).toFixed(2) : "-"} km²</div>
      </div>
    `;

    new maplibregl.Popup({
      closeButton: true,
      closeOnClick: true
    })
      .setLngLat(event.lngLat)
      .setHTML(popupHTML)
      .addTo(map);
  });

  // Ubah cursor menjadi pointer saat mouse berada di atas polygon.
  map.on("mouseenter", FILL_LAYER_ID, () => {
    map.getCanvas().style.cursor = "pointer";
  });

  map.on("mouseleave", FILL_LAYER_ID, () => {
    map.getCanvas().style.cursor = "";
  });
}

// ===============================
// 9. Event listener untuk UI
// ===============================
// Kita hanya menyediakan filter kecamatan dan tombol reset.
document.getElementById("kecamatanSelect").addEventListener("change", (event) => {
  applyKecamatanFilter(event.target.value);
});

document.getElementById("resetFilterBtn").addEventListener("click", () => {
  document.getElementById("kecamatanSelect").value = "all";
  applyKecamatanFilter("all");

  map.flyTo({
    center: [112.7521, -7.2575],
    zoom: 11
  });
});

// ===============================
// 10. Memuat data dan popup saat peta siap
// ===============================
// Karena hanya ada satu basemap, cukup memuat data setelah peta selesai dimuat.
map.on("load", () => {
  loadGeoJSON();
  setupPopup();
});



### Penjelasan Struktur `app.js`

Bagian `BASEMAP` berisi satu URL style basemap publik dari CARTO, yaitu Positron atau Light. Bagian `new maplibregl.Map()` membuat peta baru dengan container `map`, pusat koordinat Surabaya, dan zoom awal 11. Fungsi `loadGeoJSON()` membaca file `jumlah_penduduk_surabaya.geojson` dari folder `data`.

Fungsi `addPendudukLayer()` menambahkan data GeoJSON sebagai `source`, lalu membuat dua layer: layer `fill` untuk polygon dan layer `line` untuk batas wilayah. Warna polygon dibuat berdasarkan atribut `jumlah_pdd` menggunakan MapLibre expression `interpolate`. Fungsi `buildKecamatanFilter()` membaca atribut `WADMKC`, membuat daftar kecamatan unik, lalu memasukkannya ke dropdown filter.

Fungsi `applyKecamatanFilter()` menggunakan `map.setFilter()` untuk menampilkan hanya wilayah dengan kecamatan tertentu. Fungsi `setupPopup()` membuat popup ketika polygon diklik, sehingga pengguna dapat membaca nama wilayah, kecamatan, jumlah penduduk, dan luas. Event `map.on("load")` digunakan untuk menjalankan pembacaan GeoJSON dan mengaktifkan popup setelah peta siap dimuat.


## 6. Memahami File GeoJSON `jumlah_penduduk_surabaya.geojson`

File GeoJSON berisi data spasial polygon wilayah di Surabaya. Dalam tutorial ini, atribut yang digunakan adalah:

| Atribut | Fungsi |
|---|---|
| `NAMOBJ` | Nama objek/wilayah, umumnya kelurahan |
| `WADMKC` | Nama kecamatan |
| `jumlah_pdd` | Jumlah penduduk |
| `luas` | Luas wilayah |

GeoJSON dapat dibaca langsung oleh MapLibre selama formatnya valid dan koordinatnya menggunakan sistem koordinat longitude-latitude/WGS84. Pada contoh ini, data ditampilkan sebagai choropleth sederhana, yaitu polygon diberi warna berdasarkan nilai jumlah penduduk.

In [ ]:
import json
from pathlib import Path

geojson_path = Path("webgis-surabaya/data/jumlah_penduduk_surabaya.geojson")

if geojson_path.exists():
    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Jumlah fitur:", len(data["features"]))
    print("Tipe geometri fitur pertama:", data["features"][0]["geometry"]["type"])
    print("Contoh atribut fitur pertama:")
    print(data["features"][0]["properties"])
else:
    print("File GeoJSON belum ditemukan di folder data.")

## 7. Menjalankan WebGIS secara Lokal di VS Code

Cara paling mudah menjalankan WebGIS statis adalah menggunakan extension **Live Server**.

Tahapan:

1. Buka VS Code.
2. Buka folder `webgis-surabaya`.
3. Buka menu **Extensions**.
4. Cari dan install **Live Server**.
5. Klik kanan file `index.html`.
6. Pilih **Open with Live Server**.
7. Browser akan terbuka, biasanya pada alamat seperti:

```text
http://127.0.0.1:5500/index.html
```

Jangan membuka file dengan klik ganda langsung dari folder, karena `fetch("./data/jumlah_penduduk_surabaya.geojson")` dapat gagal akibat pembatasan browser terhadap file lokal. Live Server membuat server lokal sederhana sehingga file GeoJSON dapat dibaca dengan benar.

## 8. Deploy ke GitHub dari VS Code

Setelah aplikasi berjalan di lokal, tahap berikutnya adalah mengunggah project ke GitHub. Pastikan Anda sudah memiliki akun GitHub dan Git sudah terpasang di komputer.

Tahapan dasar:

1. Buka folder `webgis-surabaya` di VS Code.
2. Buka terminal VS Code melalui **Terminal > New Terminal**.
3. Jalankan perintah berikut satu per satu:

```bash
git init
git add .
git commit -m "Initial WebGIS Surabaya"
git branch -M main
git remote add origin https://github.com/USERNAME/NAMA-REPO.git
git push -u origin main
```

Ganti `USERNAME` dengan username GitHub Anda dan `NAMA-REPO` dengan nama repository yang dibuat di GitHub.

## 9. Mengaktifkan GitHub Pages agar Bisa Diakses Publik

Agar WebGIS dapat diakses publik, aktifkan GitHub Pages.

Tahapan:

1. Buka repository di GitHub.
2. Masuk ke **Settings**.
3. Pilih menu **Pages**.
4. Pada bagian **Build and deployment**, pilih:
   - **Source**: Deploy from a branch
   - **Branch**: `main`
   - **Folder**: `/root`
5. Klik **Save**.
6. Tunggu beberapa menit sampai GitHub membuat link publik.

Format link biasanya seperti:

```text
https://USERNAME.github.io/NAMA-REPO/
```

Setelah link aktif, WebGIS dapat dibuka oleh publik selama repository bersifat public atau GitHub Pages diizinkan pada akun Anda.

## 10. Checklist Akhir

Sebelum deploy, pastikan hal berikut sudah benar:

- `index.html` berada di root folder project.
- Folder `css`, `js`, dan `data` berada sejajar dengan `index.html`.
- File GeoJSON bernama tepat `jumlah_penduduk_surabaya.geojson`.
- Path di JavaScript adalah `./data/jumlah_penduduk_surabaya.geojson`.
- Aplikasi sudah berhasil dijalankan melalui Live Server.
- Dropdown kecamatan muncul.
- Popup muncul ketika polygon diklik.
- Basemap Light muncul sebagai peta dasar.
- Tidak ada lagi panel untuk mengganti basemap.

Apabila peta muncul tetapi data tidak muncul, buka **Developer Tools > Console** di browser. Kesalahan yang paling sering terjadi adalah salah nama file, salah path folder, atau file GeoJSON tidak berada di folder `data`.
